In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("training_data.csv")

# df

In [ ]:
# df.iloc[0:3]
# df.head()
# df.tail()

In [67]:
# risk["Age"].count()
risk_good = df[df["Risk"] == "good"]
risk_bad = df[df["Risk"] == "bad"]
print(len(risk_good["Age"]))
print(len(risk_bad["Age"]))


560
560


In [11]:
print(len(df[df["Job"] == 0]))
print(len(df[df["Job"] == 1]))
print(len(df[df["Job"] == 2]))
print(len(df[df["Job"] == 3]))

20
238
686
176


In [ ]:
key_job = {}
for i in df["Job"].unique():
    job = df["Job"] == i;
    good = df[(job) & (df["Risk"] == "good")]
    bad = df[(job) & (df["Risk"] == "bad")]
    score = -1 + len(good) / len(bad)
    print(f"Total with job [{i}]:  {len(df[job])}")
    print(f"    Good: {len(good)}")
    print(f"    Bad: {len(bad)}")
    print(f"    Points for job: {score:+.5f}")
    key_job[i] = score

Total with job [1]:  238
    Good: 124
    Bad: 114
    Points for job: +0.08772
Total with job [2]:  686
    Good: 340
    Bad: 346
    Points for job: -0.01734
Total with job [3]:  176
    Good: 84
    Bad: 92
    Points for job: -0.08696
Total with job [0]:  20
    Good: 12
    Bad: 8
    Points for job: +0.50000


In [ ]:
# keys = {}

# for column in df.columns:
#     if column == "ID" or column == "Risk": continue
#     for option in df[column].unique():
#         # print(f"Column: {column} Option: {option}\n")
#         this_type = df[column] == option;
#         good = df[(this_type) & (df["Risk"] == "good")]
#         bad = df[(this_type) & (df["Risk"] == "bad")]
#         # score = -1 + len(good) / (len(bad) + 0.0001)
#         g, b = len(good), len(bad)
#         score = (g - b) / (g + b + 1e-9)
#         # print(f"Total with {column} [{i}]:  {len(df[this_type])}")
#         # print(f"    Good: {len(good)}")
#         # print(f"    Bad: {len(bad)}")
#         # print(f"    Points for this {column}: {score}\n")
#         if column not in keys:
#             keys[column] = {}
#         keys[column][option] = score    

In [72]:
keys = {}

df_subset = df.iloc[:int(len(df) * 0.8)]

# print(len(df), len(df_subset), len(df) * 0.8, len(df) - len(df_subset))

for column in df_subset.columns:
    if column == "ID" or column == "Risk": continue
    for option in df_subset[column].unique():
        # print(f"Column: {column} Option: {option}\n")
        this_type = df_subset[column] == option;
        # print(df_subset)
        good = df_subset[(this_type) & (df_subset["Risk"] == "good")]
        bad = df_subset[(this_type) & (df_subset["Risk"] == "bad")]
        g, b = len(good), len(bad)
        score = (g - b) / (g + b + 1e-9)

        if column not in keys:
            keys[column] = {}
        keys[column][option] = score    

In [ ]:
print(f"Score if you are a male: {keys["Sex"]["male"]:+.5f}")
print(f"Score if you are 22 years old: {keys["Age"][22]:+.5f}")
print(f"Score if you are 50 years old: {keys["Age"][50]:+.5f}")


Score if you are a male: 0.07107438016517179
Score if you are 22 years old: -0.22580645160561916
Score if you are 50 years old: 0.5555555554938272


In [74]:
# correct = 0
# incorrect = 0

# for index, row in df.iterrows():
#     user_score = 0
#     for column in df.columns:
#         # print(df.iloc[index][column])
#         if column not in keys:
#             continue
#         score_to_add = keys[column][df.iloc[index][column]]
#         user_score += score_to_add

#     is_correct = (user_score >= 0) == (row["Risk"] == "good")
#     correct += is_correct
    
#     print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
# total_size = len(df.index)
# print(f"Correct: {correct}, incorrect: {total_size - correct}.")
# print(f"{correct / total_size * 100:.3f}% accuracy.")

In [75]:
correct = 0

for index, row in df_subset.iterrows():
    user_score = 0
    for column in df_subset.columns:
        # print(df_subset.iloc[index][column])
        if column not in keys:
            continue
        score_to_add = keys[column].get(row[column], 0)
        user_score += score_to_add

    is_correct = (user_score >= 0) == (row["Risk"] == "good")
    correct += is_correct
    
    # print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
total_size = len(df_subset.index)
print(f"Total rows: {total_size}")
print(f"Correct: {correct}, incorrect: {total_size - correct}.")
print(f"{correct / total_size * 100:.3f}% accuracy.")

Total rows: 896
Correct: 876, incorrect: 20.
97.768% accuracy.


In [76]:
df_subset_tail = df.iloc[int(len(df) * 0.8):].reset_index(drop=True)
correct = 0

for index, row in df_subset_tail.iterrows():
    user_score = 0
    for column in df_subset_tail.columns:
        if column not in keys:
            continue
        # score_to_add = keys[column][row[column]]
        score_to_add = keys[column].get(row[column], 0)
        user_score += score_to_add

    is_correct = (user_score >= 0) == (row["Risk"] == "good")
    correct += is_correct
    
    # print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
total_size = len(df_subset_tail.index)
print(f"Total rows: {total_size}")
print(f"Correct: {correct}, incorrect: {total_size - correct}.")
print(f"{correct / total_size * 100:.3f}% accuracy.")

Total rows: 224
Correct: 166, incorrect: 58.
74.107% accuracy.


In [83]:
tf = pd.read_csv("test_data.csv")
rejected = 0
for index, row in tf.iterrows():
    user_score = 0
    for column in tf.columns:
        if column not in keys:
            continue
        score_to_add = keys[column][df.iloc[index][column]]
        user_score += score_to_add

    rejected += user_score >= 0
    
    print(f"Calculated risk for this individual: {"good" if user_score >= 0 else "bad"}, risk score evaluated as: {user_score:+.3f}")

total_size = len(df.index)
print(f"Training data 'bad' risk percent: {((total_size - len(df[df["Risk"] == "bad"])) / total_size) * 100:.1f}%")
total_size_test = len(tf.index)
print(f"Testing data 'bad' risk percent: {((rejected) / total_size_test) * 100:.1f}%")
print(f"good : bad -> {total_size_test - rejected} : {rejected}")


Calculated risk for this individual: bad, risk score evaluated as: -1.924
Calculated risk for this individual: good, risk score evaluated as: +0.510
Calculated risk for this individual: good, risk score evaluated as: +1.413
Calculated risk for this individual: good, risk score evaluated as: +1.132
Calculated risk for this individual: bad, risk score evaluated as: -1.058
Calculated risk for this individual: good, risk score evaluated as: +0.883
Calculated risk for this individual: bad, risk score evaluated as: -1.174
Calculated risk for this individual: bad, risk score evaluated as: -1.299
Calculated risk for this individual: bad, risk score evaluated as: -2.345
Calculated risk for this individual: bad, risk score evaluated as: -0.981
Calculated risk for this individual: bad, risk score evaluated as: -1.685
Calculated risk for this individual: bad, risk score evaluated as: -1.843
Calculated risk for this individual: bad, risk score evaluated as: -0.658
Calculated risk for this individua

In [79]:
print(f"Males have {keys["Sex"]["male"]:+.3f} ({len(df[(df["Sex"] == "male") & (df["Risk"] == "good")])} / {len(df[(df["Sex"] == "male")])} = {(len(df[(df["Sex"] == "male") & (df["Risk"] == "good")]) / len(df[(df["Sex"] == "male")])):.3f}).")
print(f"Females have {keys["Sex"]["female"]:+.3f} ({len(df[(df["Sex"] == "female") & (df["Risk"] == "good")])} / {len(df[(df["Sex"] == "female")])} = {(len(df[(df["Sex"] == "female") & (df["Risk"] == "good")]) / len(df[(df["Sex"] == "female")])):.3f}).")
print(f"That is a difference of {abs(keys["Sex"]["female"] - keys["Sex"]["male"]):.3f}")

Males have +0.071 (397 / 748 = 0.531).
Females have -0.127 (163 / 372 = 0.438).
That is a difference of 0.198
